In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os, asyncio, uuid, requests
import psycopg
from openai import AzureOpenAI
from typing import Annotated
from pydantic import Field
from dotenv import load_dotenv
from IPython.display import Markdown
import time

In [ ]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT   = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_DEPLOYMENT = os.environ["AZURE_OPENAI_DEPLOYMENT"]
AZURE_OPENAI_KEY        = os.environ["AZURE_OPENAI_KEY"]
AZURE_EMBED_DEPLOYMENT  = os.environ["AZURE_EMBED_DEPLOYMENT"]
AZURE_API_VERSION       = os.environ["AZURE_API_VERSION"]

DB_CONFIG = {
    "host":     os.environ["AZURE_PG_HOST"],
    "dbname":   os.environ["AZURE_PG_NAME"],
    "user":     os.environ["AZURE_PG_USER"],
    "password": os.environ["AZURE_PG_PASSWORD"],
    "port":     os.environ["AZURE_PG_PORT"],
    "sslmode":  os.environ.get("AZURE_PG_SSLMODE", "require"),
}

print(AZURE_OPENAI_ENDPOINT)
print(AZURE_OPENAI_DEPLOYMENT)
print(AZURE_OPENAI_KEY)
print(AZURE_EMBED_DEPLOYMENT)
print(AZURE_API_VERSION)
print(DB_CONFIG)

In [ ]:
# Initialize dataset: create tables and load cases from CSV
import os

csv_path = os.path.join(os.path.dirname(os.getcwd()), "Dataset", "cases.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(os.getcwd(), "Dataset", "cases.csv")
print(f"CSV path: {csv_path}")

conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    # Drop existing tables
    cur.execute("DROP TABLE IF EXISTS cases;")
    cur.execute("DROP TABLE IF EXISTS temp_cases;")

    # Create the cases table
    cur.execute("""
        CREATE TABLE cases(
            id SERIAL PRIMARY KEY,
            name TEXT,
            decision_date DATE,
            court_level TEXT,
            opinion TEXT
        );
    """)

    # Create temp staging table and load CSV
    cur.execute("CREATE TABLE temp_cases(data jsonb);")

    with open(csv_path, "r", encoding="utf-8") as f:
        with cur.copy("COPY temp_cases (data) FROM STDIN WITH (FORMAT csv, HEADER true)") as copy:
            while chunk := f.read(8192):
                copy.write(chunk)

    print("CSV loaded into temp_cases.")

    # Insert parsed data into cases table
    cur.execute("""
        INSERT INTO cases
        SELECT
            (data#>>'{id}')::int AS id,
            (data#>>'{name_abbreviation}')::text AS name,
            (data#>>'{decision_date}')::date AS decision_date,
            (data#>>'{court,name}')::text AS court_level,
            array_to_string(
                ARRAY(SELECT jsonb_path_query(data, '$.casebody.opinions[*].text')),
                ', '
            ) AS opinion
        FROM temp_cases;
    """)

    cur.execute("SELECT count(*) FROM cases;")
    count = cur.fetchone()[0]
    print(f"Done. {count} cases loaded into the cases table.")

conn.close()

In [ ]:
# Test query: select top 5 rows from cases
conn = psycopg.connect(**DB_CONFIG)
with conn.cursor() as cur:
    cur.execute("SELECT * FROM cases LIMIT 5;")
    rows = cur.fetchall()
    for row in rows:
        print(row)
conn.close()

In [ ]:
# Add the vector column to the cases table
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True
with conn.cursor() as cur:
    cur.execute("ALTER TABLE cases ADD COLUMN IF NOT EXISTS opinions_vector vector(1536);")    
conn.close()

print("opinions_vector column added.")

In [ ]:
BATCH_SIZE = 20
CLEAR_VECTORS_BEFORE_RUN = True  # Set to False to skip clearing existing vectors

# Azure OpenAI client for embeddings
embed_client = AzureOpenAI(
    api_key=AZURE_OPENAI_KEY,
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)

# Connect and fetch rows that still need embeddings
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

# Optionally clear all existing vectors
if CLEAR_VECTORS_BEFORE_RUN:
    with conn.cursor() as cur:
        cur.execute("UPDATE cases SET opinions_vector = NULL")
    print("Cleared all existing vectors.")

start_time = time.time()
print(f"Start time: {time.strftime('%H:%M:%S', time.localtime(start_time))}")

with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM cases WHERE opinions_vector IS NULL")
    total_rows = cur.fetchone()[0]
    print(f"Total rows to process: {total_rows}")

    cur.execute("SELECT id, name, opinion FROM cases WHERE opinions_vector IS NULL ORDER BY id")
    rows = cur.fetchall()

processed = 0
for i in range(0, len(rows), BATCH_SIZE):
    batch = rows[i : i + BATCH_SIZE]
    texts = [(name or "") + " " + (opinion or "")[:8000] for _id, name, opinion in batch]

    # Call Azure OpenAI embeddings
    response = embed_client.embeddings.create(
        input=texts,
        model=AZURE_EMBED_DEPLOYMENT,
    )

    # Update each row with its embedding
    with conn.cursor() as cur:
        for j, item in enumerate(response.data):
            vector_str = "[" + ",".join(str(v) for v in item.embedding) + "]"
            cur.execute(
                "UPDATE cases SET opinions_vector = %s::vector WHERE id = %s",
                (vector_str, batch[j][0]),
            )

    processed += len(batch)
    pct = round(100.0 * processed / total_rows, 1)
    print(f"Progress: {processed} / {total_rows} ({pct}%)")

conn.close()

end_time = time.time()
duration = end_time - start_time
print(f"\nEnd time: {time.strftime('%H:%M:%S', time.localtime(end_time))}")
print(f"Total duration: {int(duration // 60)}m {duration % 60:.1f}s")
print(f"Done. {processed} rows processed.")

### Create Graph

In [ ]:
# Create the case citation graph using Apache AGE
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    # 1) Create the two helper functions
    cur.execute("""
        CREATE OR REPLACE FUNCTION create_case_in_case_graph(case_id text)
        RETURNS void
        LANGUAGE plpgsql
        VOLATILE
        AS $BODY$
        BEGIN
            SET search_path TO ag_catalog;
            EXECUTE format(
                'SELECT * FROM cypher(''case_graph'', $$CREATE (:case {case_id: %s})$$) AS (a agtype);',
                quote_ident(case_id)
            );
        END
        $BODY$;
    """)

    cur.execute("""
        CREATE OR REPLACE FUNCTION create_case_link_in_case_graph(id_from text, id_to text)
        RETURNS void
        LANGUAGE plpgsql
        VOLATILE
        AS $BODY$
        BEGIN
            SET search_path TO ag_catalog;
            EXECUTE format(
                'SELECT * FROM cypher(''case_graph'', $$MATCH (a:case), (b:case) WHERE a.case_id = %s AND b.case_id = %s CREATE (a)-[e:REF]->(b) RETURN e$$) AS (a agtype);',
                quote_ident(id_from), quote_ident(id_to)
            );
        END
        $BODY$;
    """)
    print("Helper functions created.")

    # 2) Install AGE extension
    cur.execute("CREATE EXTENSION IF NOT EXISTS age CASCADE;")
    print("AGE extension ready.")

    # 3) Drop existing graph if it exists (graceful - no error if missing)
    cur.execute("""
        DO $$
        BEGIN
            IF EXISTS (SELECT 1 FROM ag_catalog.ag_graph WHERE name = 'case_graph') THEN
                PERFORM ag_catalog.drop_graph('case_graph', true);
            END IF;
        END $$;
    """)
    print("Old case_graph dropped (if it existed).")

    # 4) Create the graph
    cur.execute("SET search_path = public, ag_catalog, \"$user\";")
    cur.execute("SELECT create_graph('case_graph');")
    print("case_graph created.")

    # 5) Create nodes from temp_cases
    cur.execute("""
        SELECT create_case_in_case_graph((public.temp_cases.data#>>'{id}')::text)
        FROM public.temp_cases;
    """)
    print("Graph nodes created.")

    # 6) Verify node count
    cur.execute("""
        SELECT * FROM cypher('case_graph', $$
            MATCH (n)
            RETURN COUNT(n.case_id)
        $$) AS (case_id TEXT);
    """)
    node_count = cur.fetchone()[0]
    print(f"Node count: {node_count}")

    # 7) Create citation edges
    cur.execute("""
        WITH edges AS (
            SELECT c1.data#>>'{id}' AS id_from, c2.data#>>'{id}' AS id_to
            FROM public.temp_cases c1
            LEFT JOIN
                LATERAL jsonb_array_elements(c1.data -> 'cites_to') AS cites_to_element ON true
            LEFT JOIN
                LATERAL jsonb_array_elements(cites_to_element -> 'case_ids') AS case_ids ON true
            JOIN public.temp_cases c2
                ON case_ids::text = c2.data#>>'{id}'
        )
        SELECT public.create_case_link_in_case_graph(edges.id_from::text, edges.id_to::text)
        FROM edges;
    """)
    print("Citation edges created.")

conn.close()
print("Done. Case graph is ready.")

In [ ]:
conn = psycopg.connect(**DB_CONFIG)
conn.autocommit = True

with conn.cursor() as cur:
    # Drop existing tables
    cur.execute("CREATE EXTENSION IF NOT EXISTS pg_fts;")
    cur.execute("SET search_path = public, pgfts;")
    cur.execute("SELECT hello_pg_fts();")

    result = cur.fetchone()

    print(f"pg_fts extension is ready. Result: {result}")
   
    cur.execute("DROP INDEX IF EXISTS idx_cases_fts;")
    cur.execute("""
        CREATE INDEX idx_cases_fts
        ON public.cases
        USING fts (name text_fts_ops, opinion text_fts_ops);
        """)
    
    cur.execute("""
        SELECT count(*) FROM cases
        WHERE fts_query('water leaking', 'idx_cases_fts')
        """)
    
    count = cur.fetchone()[0]

    print(f"fts_query executed. Matching cases: {count}")

conn.close()

In [ ]:
import psycopg

with psycopg.connect(**DB_CONFIG, autocommit=True) as conn, conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cur.execute("CREATE EXTENSION IF NOT EXISTS pg_diskann;")

    # DiskANN index (quantization not supported yet on HorizonDB)
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_cases_diskann
        ON public.cases
        USING diskann (opinions_vector vector_cosine_ops)
        WITH (product_quantized='false');
    """)
print("pg_diskann ready: idx_cases_diskann on public.cases(opinions_vector)")